# 🎬 keepsfx → giữ lại HIỆU ỨNG (SFX)

Bỏ giọng + nhạc khỏi video/audio, giữ lại tiếng động/hiệu ứng → MP4 + WAV để lồng tiếng Việt.
Dùng **BandIt** (SOTA, license CC BY-NC — phi thương mại).

**Backup tự động:** sau mỗi chunk xong → lưu Drive (`keepsfx_progress/`) → nếu Colab hết hạn, lần sau mở lại tự tiếp tục.

**Bước duy nhất:** chọn `Runtime → GPU T4 → Save`, rồi bấm ▶ chạy từng ô theo thứ tự.

In [ ]:
import subprocess, os, importlib.util, shutil

# === GPU ===
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print(f'GPU: {r.stdout.strip()}')
print('Uoc tinh (bark48, chunk 60s): T4~1h20-1h50p | V100~50p-1h10p | A100~25-35p (Colab Pro)')
print('Video > 1h tren T4: doi MODEL_STEM = erb48 de nhanh ~2x')
print()

print('=== BUOC 1: Mount Drive ===', flush=True)
from google.colab import drive
drive.mount('/content/drive')

print('=== BUOC 2: Clone code ===', flush=True)
os.chdir('/content')
!rm -rf /content/keepsfx /content/bandit
!git clone -q https://github.com/yaylbanh/keepsfx.git
!git clone -q https://github.com/kwatcharasupat/bandit.git

print('=== BUOC 3: Cai thu vien (lan dau lau vai phut) ===', flush=True)
if not all(importlib.util.find_spec(m) for m in ['gradio', 'lightning', 'asteroid', 'fire', 'librosa']):
    !pip install -r /content/keepsfx/requirements.txt
    print('=== Cai xong ===', flush=True)
else:
    print('=== Thu vien da co, bo qua ===', flush=True)

print('=== BUOC 4: Chuan bi model BandIt ===', flush=True)

# ============================================================
# CHON MODEL:
#   'dnr-3s-erb48-l1snr'  <- NHANH NHAT (~2x, chat luong hoi kem) <- DUNG CHO VIDEO DAI
#   'dnr-3s-mus64-l1snr'  <- NHANH      (~1.5x)
#   'dnr-3s-bark48-l1snr' <- CAN BANG   (mac dinh)
#   'dnr-3s-bark64-l1snr' <- CHAT LUONG (rat cham tren T4, khong khuyen dung)
# ============================================================
MODEL_STEM = 'dnr-3s-bark48-l1snr'

MODELS   = '/content/drive/MyDrive/keepsfx_models'
CKPT_DIR = os.path.join(MODELS, 'ckpt')
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT = os.path.join(CKPT_DIR, MODEL_STEM + '.ckpt')
if not os.path.isfile(CKPT):
    print(f'[*] Tai {MODEL_STEM} (~775MB, luu Drive de lan sau khoi tai)...', flush=True)
    !wget -q --show-progress -O "{CKPT}" "https://zenodo.org/records/10160698/files/{MODEL_STEM}.ckpt?download=1"
else:
    print(f'[*] Checkpoint {MODEL_STEM} da co.', flush=True)
shutil.copyfile(f'/content/bandit/expt/{MODEL_STEM}.yaml', os.path.join(MODELS, 'hparams.yaml'))

print('=== BUOC 5: Khoi dong app ===', flush=True)
os.environ['BANDIT_DIR']       = '/content/bandit'
os.environ['BANDIT_CKPT']      = CKPT
os.environ['KEEPSFX_OUTPUT']   = '/content/drive/MyDrive/keepsfx_output'
os.environ['KEEPSFX_PROGRESS'] = '/content/drive/MyDrive/keepsfx_progress'
os.environ['KEEPSFX_SHARE']    = '0'
os.environ['KEEPSFX_CHUNK_SEC']= '60'   # 60s: an toan voi T4; tang 90 neu dung A100/V100
os.chdir('/content/keepsfx')
%run app.py